In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
!nvidia-smi

CUDA available: True
Tue Sep 29 06:48:44 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P8             13W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+--------------------------

In [ ]:
!pip install -q bitsandbytes peft trl datasets transformers accelerate matplotlib scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 64.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 925.0/925.0 kB 62.5 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!cp -r /content/drive/MyDrive/exp_c_adapters/* /content/
!ls -d /content/phi3-vit-lora-expc*
!ls /content/phi3-vit-lora-expc-idk0        # must show adapter_config.json + adapter_model.safetensors

/content/phi3-vit-lora-expc-idk0   /content/phi3-vit-lora-expc-idk15
/content/phi3-vit-lora-expc-idk10
adapter_config.json	   checkpoint-400	  tokenizer.json
adapter_model.safetensors  checkpoint-600	  training_args.bin
chat_template.jinja	   README.md
checkpoint-200		   tokenizer_config.json


In [ ]:
!ls eval_compare_fixed.py expc_idk15_test.jsonl

eval_compare_fixed.py  expc_idk15_test.jsonl


In [ ]:
!python eval_compare_fixed.py --normal-adapter phi3-vit-lora-expc-idk0 --idk-adapter phi3-vit-lora-expc-idk15 --test expc_idk15_test.jsonl

449 examples (79 IDK, 370 answerable)

Loading base model...
Loading weights: 100% 195/195 [00:02<00:00, 79.78it/s]
Evaluating base: 100% 449/449 [28:17<00:00,  3.78s/ex]
Loading normal fine-tune (phi3-vit-lora-expc-idk0)...
Loading weights: 100% 195/195 [00:02<00:00, 87.84it/s]
Evaluating normal-ft: 100% 449/449 [22:31<00:00,  3.01s/ex]
Loading IDK fine-tune (phi3-vit-lora-expc-idk15)...
Loading weights: 100% 195/195 [00:02<00:00, 86.80it/s]
Evaluating idk-ft: 100% 449/449 [22:05<00:00,  2.95s/ex]
  Comparison (F1>=0.5): Base vs Normal FT vs IDK FT
Metric                Base                Normal FT           IDK FT              
--------------------------------------------------------------------------------------------
Accuracy (semantic)   0.100               0.327               0.477               
Exact Match           0.000               0.051               0.189               
Token F1              0.168               0.391               0.515               
ECE                

In [ ]:
!python run_calibration_utils.py --input results/calibration_export.json

Models in export: ['Base', 'Normal FT', 'IDK FT']

=== Base ===
  Saved ROC curve to results/roc_curve.png
  Saved reliability diagram to results/reliability_diagram_base.png
  AUROC (hallucination detection): 0.557
  ECE (via calibration_eval_utils): 0.622

=== Normal FT ===
  Saved ROC curve to results/roc_curve.png
  Saved reliability diagram to results/reliability_diagram_normal_ft.png
  AUROC (hallucination detection): 0.764
  ECE (via calibration_eval_utils): 0.408

=== IDK FT ===
  Saved ROC curve to results/roc_curve.png
  Saved reliability diagram to results/reliability_diagram_idk_ft.png
  AUROC (hallucination detection): 0.810
  ECE (via calibration_eval_utils): 0.304

Note on AUROC here: it measures whether the model's OWN confidence
score can tell apart its correct answers from its wrong ones -- a
high AUROC means low confidence is a good warning sign of a bad
answer, even before you know the true answer. This is a different
question from ECE (is confidence numerically wel

In [ ]:
!mkdir -p /content/drive/MyDrive/exp_c_results_after_accuracy
!cp -r results/* /content/drive/MyDrive/exp_c_results_after_accuracy/

In [ ]:
!python compare_idk_stats.py --normal-adapter phi3-vit-lora-expc-idk0 --idk-adapter phi3-vit-lora-expc-idk15 --test expc_idk15_test.jsonl

Test entries: 449  (IDK: 79, Answerable: 370)

Loading BASE model...
Loading weights: 100% 195/195 [00:02<00:00, 79.46it/s]
Evaluating base: 100% 449/449 [35:58<00:00,  4.81s/ex]

  -- BASE --------------------------------
  Refusal rate          : 0.345 (155/449)
  IDK precision         : 0.252  (of refusals, how many were correct)
  IDK recall            : 0.494  (of IDK questions, how many were refused)
  IDK F1                : 0.333
  False refusals (FP)   : 116  (refused answerable questions)
  Missed IDK (FN)       : 40  (answered IDK questions confidently)
  Acc on answerable     : 0.227  (word-overlap >= 0.4)

Loading NORMAL fine-tuned model (phi3-vit-lora-expc-idk0)...
Loading weights: 100% 195/195 [00:02<00:00, 84.94it/s]
Evaluating normal-ft: 100% 449/449 [20:42<00:00,  2.77s/ex]

  -- NORMAL-FT --------------------------------
  Refusal rate          : 0.000 (0/449)
  IDK precision         : 0.000  (of refusals, how many were correct)
  IDK recall            : 0.000  (of I

In [ ]:
!mkdir -p results_idk15_backup_after_accuracy !cp results/*.md results/*.png results/*.json results_idk15_backup_after_accuracy/ 2>/dev/null !cp error_examples_normal_ft.csv error_examples_idk_ft.csv results_idk15_backup_after_accuracy/ 2>/dev/null

In [ ]:
!python run_calibration_utils.py --input results/calibration_export.json

Models in export: ['Base', 'Normal FT', 'IDK FT']

=== Base ===
  Saved ROC curve to results/roc_curve.png
  Saved reliability diagram to results/reliability_diagram_base.png
  AUROC (hallucination detection): 0.557
  ECE (via calibration_eval_utils): 0.622

=== Normal FT ===
  Saved ROC curve to results/roc_curve.png
  Saved reliability diagram to results/reliability_diagram_normal_ft.png
  AUROC (hallucination detection): 0.764
  ECE (via calibration_eval_utils): 0.408

=== IDK FT ===
  Saved ROC curve to results/roc_curve.png
  Saved reliability diagram to results/reliability_diagram_idk_ft.png
  AUROC (hallucination detection): 0.810
  ECE (via calibration_eval_utils): 0.304

Note on AUROC here: it measures whether the model's OWN confidence
score can tell apart its correct answers from its wrong ones -- a
high AUROC means low confidence is a good warning sign of a bad
answer, even before you know the true answer. This is a different
question from ECE (is confidence numerically wel

In [ ]:
!mkdir -p /content/drive/MyDrive/exp_c_results_after_accuracy
!cp -r results/* /content/drive/MyDrive/exp_c_results_after_accuracy/

In [ ]:
!pip install -q scikit-learn matplotlib
!python make_figures.py

wrote reliability.png
wrote roc.png
wrote confidence_hist.png
Done -> results/figures/


In [ ]:
!mkdir -p /content/drive/MyDrive/exp_c_results_after_accuracy
!cp -r results/* /content/drive/MyDrive/exp_c_results_after_accuracy/

In [ ]:
!python eval_compare_fixed.py \
  --normal-adapter /content/phi3-vit-lora-expc-idk0 \
  --idk-adapter    /content/phi3-vit-lora-expc-idk15 \
  --test           /content/expc_idk15_test.jsonl \
  --first-sentence

449 examples (79 IDK, 370 answerable) | first-sentence trimming ON

Loading base model...
Loading weights: 100% 195/195 [00:02<00:00, 84.38it/s]
Evaluating base: 100% 449/449 [28:25<00:00,  3.80s/ex]
Loading normal fine-tune (/content/phi3-vit-lora-expc-idk0)...
Loading weights: 100% 195/195 [00:02<00:00, 88.78it/s]
Evaluating normal-ft: 100% 449/449 [23:02<00:00,  3.08s/ex]
Loading IDK fine-tune (/content/phi3-vit-lora-expc-idk15)...
Loading weights: 100% 195/195 [00:02<00:00, 88.96it/s]
Evaluating idk-ft: 100% 449/449 [22:48<00:00,  3.05s/ex]
  Comparison (first-sentence, F1>=0.5): Base vs Normal FT vs IDK FT
Metric                Base                Normal FT           IDK FT              
--------------------------------------------------------------------------------------------
Accuracy (semantic)   0.136               0.307               0.468               
Exact Match           0.000               0.045               0.189               
Token F1              0.171            